<a href="https://colab.research.google.com/github/JaquelineBrandao/Projeto_Conversando_por_Voz/blob/main/desafio_dio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. instalando dependencias

In [ ]:
!pip install openai gtts streamlit streamlit-audiorecorder python-dotenv pydub google-generativeai
!pip install --upgrade google-generativeai # Garante a versão mais recente

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.7/487.7 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 88.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


2. Configurando chave api


In [ ]:
from google.colab import userdata
import os

# Altere os nomes dos segredos para chaves descritivas que você definiu no Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') # Certifique-se de que 'OPENAI_API_KEY' está definido nos seus Segredos do Colab
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY') # Certifique-se de que 'GOOGLE_API_KEY' está definido nos seus Segredos do Colab

print("Chaves de API carregadas do Colab Secrets.")

Chaves de API carregadas do Colab Secrets.


3: Definir Funções de Serviço

In [ ]:
import openai
import google.generativeai as genai # Renomeado para evitar conflito com google.genai
from gtts import gTTS
import os
import tempfile
from pydub import AudioSegment # Necessário para o audiorecorder exportar para .wav
import time # Importar time para backoff exponencial

# Configurações iniciais
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# --- Funções de Serviço ---

def transcrever_audio(caminho_audio: str, idioma: str = None) -> str:
    """
    Transcreve o áudio usando o modelo Whisper da OpenAI.
    Se idioma for None, o Whisper detecta automaticamente.
    """
    max_retries = 3
    for attempt in range(max_retries):
        try:
            with open(caminho_audio, "rb") as arquivo_audio:
                params = {
                    "model": "whisper-1",
                    "file": arquivo_audio,
                }
                if idioma:
                    params["language"] = idioma

                transcricao = openai_client.audio.transcriptions.create(**params)
            return transcricao.text # Retorna em caso de sucesso
        except openai.RateLimitError:
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) + 1 # Backoff exponencial com jitter
                print(f"OpenAI Whisper RateLimitError: Tentando novamente em {wait_time} segundos...")
                time.sleep(wait_time)
            else:
                print("OpenAI Whisper RateLimitError: Todas as tentativas de reenvio falharam.")
                return "Desculpe, o serviço de transcrição de áudio da OpenAI está sobrecarregado no momento. Por favor, tente novamente mais tarde."
        except Exception as e:
            print(f"Erro ao transcrever áudio com OpenAI Whisper: {e}")
            return "Desculpe, ocorreu um erro ao transcrever o áudio. Por favor, tente novamente."

def obter_resposta_llm(model_choice: str, texto: str, historico: list = []) -> tuple[str, list]:
    """
    Envia o texto ao modelo de LLM escolhido (Gemini ou OpenAI) e retorna a resposta + histórico atualizado.
    O histórico permite manter o contexto da conversa.
    """
    mensagem_resposta = ""

    if model_choice == "Gemini":
        model = genai.GenerativeModel('gemini-pro')
        gemini_historico = []
        for msg in historico:
            role = "user" if msg["role"] == "user" else "model"
            gemini_historico.append({"role": role, "parts": [msg["content"]]})

        try:
            chat = model.start_chat(history=gemini_historico)
            response = chat.send_message(texto)
            mensagem_resposta = response.text
        except Exception as e:
            print(f"Erro ao chamar Gemini: {e}")
            mensagem_resposta = "Desculpe, não consegui gerar uma resposta no momento usando Gemini. Tente novamente."

    elif model_choice == "OpenAI":
        openai_messages = []
        # Converter histórico para o formato do OpenAI
        for msg in historico:
            openai_role = "user" if msg["role"] == "user" else "assistant"
            openai_messages.append({"role": openai_role, "content": msg["content"]})

        # Adicionar a mensagem atual do usuário
        openai_messages.append({"role": "user", "content": texto})

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = openai_client.chat.completions.create(
                    model="gpt-3.5-turbo", # Pode ser alterado para "gpt-4" se desejar
                    messages=openai_messages,
                )
                mensagem_resposta = response.choices[0].message.content
                break # Sai do loop se a chamada for bem-sucedida
            except openai.RateLimitError:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + 1 # Backoff exponencial com jitter
                    print(f"OpenAI LLM RateLimitError: Tentando novamente em {wait_time} segundos...")
                    time.sleep(wait_time)
                else:
                    mensagem_resposta = "Desculpe, a OpenAI está sobrecarregada no momento. Por favor, tente novamente mais tarde."
                    print("OpenAI LLM RateLimitError: Todas as tentativas de reenvio falharam.")
            except Exception as e:
                print(f"Erro ao chamar OpenAI LLM: {e}")
                mensagem_resposta = "Desculpe, não consegui gerar uma resposta no momento usando OpenAI. Tente novamente."
                break # Sai do loop para outros erros
    else:
        mensagem_resposta = "Modelo de IA não reconhecido. Por favor, escolha 'Gemini' ou 'OpenAI'."

    # Atualiza o histórico para o formato do nosso app
    historico.append({"role": "user", "content": texto})
    historico.append({"role": "assistant", "content": mensagem_resposta})

    return mensagem_resposta, historico

def texto_para_voz(texto: str, idioma: str = "pt") -> str:
    """
    Converte texto em áudio usando gTTS.
    Retorna o caminho do arquivo de áudio gerado.
    """
    tts = gTTS(text=texto, lang=idioma, slow=False)

    arquivo_temp = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp3"
    )
    tts.save(arquivo_temp.name)

    return arquivo_temp.name

def detectar_idioma_gtts(texto: str) -> str:
    """
    Detecta o idioma do texto para passar ao gTTS.
    Lógica simples por palavras-chave — pode ser expandida.
    """
    texto_lower = texto.lower()

    if any(p in texto_lower for p in ["the", "is", "are", "hello", "what"]):
        return "en"
    elif any(p in texto_lower for p in ["el", "la", "los", "hola", "que"]):
        return "es"
    elif any(p in texto_lower for p in ["le", "les", "bonjour", "est", "une"]):
        return "fr"
    else:
        return "pt"  # padrão: português

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: 

4: Executar o Aplicativo Streamlit


In [ ]:
# Instalar ngrok
!pip install pyngrok

In [ ]:
# Instalar ngrok
!pip install pyngrok

from pyngrok import ngrok
import streamlit as st
import os
import tempfile
from audiorecorder import audiorecorder # Importar aqui

from google.colab import userdata

# --- Configuração da página ─────────────────────────────────────────────
st.set_page_config(
    page_title="Voice AI Assistant",
    page_icon="🎤",
    layout="centered"
)

st.title("🎤 Voice AI Assistant")
st.caption("Fale com seu assistente de IA em qualquer idioma — powered by Whisper + gTTS + Gemini/OpenAI")

# Model selection
st.sidebar.header("Configurações do Modelo")
model_choice = st.sidebar.radio(
    "Escolha o Modelo de IA:",
    ("Gemini", "OpenAI"),
    index=0, # Default to Gemini
    key="llm_model_choice"
)

# --- Estado da sessão ─────────────────────────────────────────────
if "historico" not in st.session_state:
    st.session_state.historico = []

if "mensagens_exibidas" not in st.session_state:
    st.session_state.mensagens_exibidas = []

# --- Exibir histórico de mensagens ──────────────────────────────────────
for msg in st.session_state.mensagens_exibidas:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# --- Gravação de áudio ──────────────────────────────────────────
st.divider()
st.subheader("🎤 Grave sua pergunta")

# O audiorecorder pode precisar de um pouco mais de tempo para carregar no Colab
# ou pode ter problemas de compatibilidade com alguns navegadores/ambientes.
# Se tiver problemas, considere uma alternativa de upload de arquivo de áudio.
audio = audiorecorder("▶ Clique para gravar", "■ Clique para parar")

if len(audio) > 0:
    # Salva o áudio gravado em arquivo temporário
    # O audiorecorder retorna um objeto AudioSegment do pydub, precisamos exportar
    with tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".wav"
    ) as tmp:
        audio.export(tmp.name, format="wav") # Exporta o AudioSegment para .wav
        caminho_audio = tmp.name

    st.audio(caminho_audio, format="audio/wav")

    with st.spinner("🔍 Transcrevendo sua fala..."):
        texto_usuario = transcrever_audio(caminho_audio)

    st.success(f"**Você disse:** {texto_usuario}")

    # Adiciona ao histórico exibido
    st.session_state.mensagens_exibidas.append({
        "role": "user",
        "content": texto_usuario
    })

    with st.spinner("🤖 Assistente pensando..."):
        resposta_texto, st.session_state.historico = obter_resposta_llm( # Changed function name
            model_choice, # New argument
            texto_usuario,
            st.session_state.historico
        )

    # Adiciona resposta ao histórico exibido
    st.session_state.mensagens_exibidas.append({
        "role": "assistant",
        "content": resposta_texto
    })

    with st.chat_message("assistant"):
        st.write(resposta_texto)

    # Converte resposta em áudio
    with st.spinner("🔊 Gerando áudio da resposta..."):
        idioma_tts = detectar_idioma_gtts(resposta_texto)
        caminho_resposta_audio = texto_para_voz(resposta_texto, idioma=idioma_tts)

    st.audio(caminho_resposta_audio, format="audio/mp3", autoplay=True)

    # Limpa os arquivos temporários
    os.unlink(caminho_audio)
    os.unlink(caminho_resposta_audio)

# --- Botão para limpar conversa ──────────────────────────────────
st.divider()
if st.button("🗑️ Limpar conversa"):
    st.session_state.historico = []
    st.session_state.mensagens_exibidas = []
    st.rerun()

# --- Executar Streamlit com ngrok ---
# O Streamlit precisa ser executado em um processo separado para que o ngrok funcione.
# Vamos usar um truque para rodar o Streamlit em background e o ngrok em foreground.

# Salva o script Streamlit em um arquivo temporário
streamlit_script = """
import streamlit as st
import os
import tempfile
from audiorecorder import audiorecorder
from pydub import AudioSegment # Importar aqui também para o script salvo
import time # Adicionar import para time

# --- Funções de Serviço (copiadas para o script para evitar problemas de importação) ---
import openai
import google.generativeai as genai
from gtts import gTTS

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def transcrever_audio(caminho_audio: str, idioma: str = None) -> str:
    max_retries = 3
    for attempt in range(max_retries):
        try:
            with open(caminho_audio, "rb") as arquivo_audio:
                params = {
                    "model": "whisper-1",
                    "file": arquivo_audio,
                }
                if idioma:
                    params["language"] = idioma
                transcricao = openai_client.audio.transcriptions.create(**params)
            return transcricao.text
        except openai.RateLimitError:
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) + 1
                print(f"OpenAI Whisper RateLimitError: Tentando novamente em {wait_time} segundos...")
                time.sleep(wait_time)
            else:
                print("OpenAI Whisper RateLimitError: Todas as tentativas de reenvio falharam.")
                return "Desculpe, o serviço de transcrição de áudio da OpenAI está sobrecarregado no momento. Por favor, tente novamente mais tarde."
        except Exception as e:
            print(f"Erro ao transcrever áudio com OpenAI Whisper: {e}")
            return "Desculpe, ocorreu um erro ao transcrever o áudio. Por favor, tente novamente."

def obter_resposta_llm(model_choice: str, texto: str, historico: list = []) -> tuple[str, list]:
    mensagem_resposta = ""

    if model_choice == "Gemini":
        model = genai.GenerativeModel('gemini-pro')
        gemini_historico = []
        for msg in historico:
            role = "user" if msg["role"] == "user" else "model"
            gemini_historico.append({"role": role, "parts": [msg["content"]]})

        try:
            chat = model.start_chat(history=gemini_historico)
            response = chat.send_message(texto)
            mensagem_resposta = response.text
        except Exception as e:
            print(f"Erro ao chamar Gemini: {e}")
            mensagem_resposta = "Desculpe, não consegui gerar uma resposta no momento usando Gemini. Tente novamente."

    elif model_choice == "OpenAI":
        openai_messages = []
        for msg in historico:
            openai_role = "user" if msg["role"] == "user" else "assistant"
            openai_messages.append({"role": openai_role, "content": msg["content"]})

        openai_messages.append({"role": "user", "content": texto})

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = openai_client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=openai_messages,
                )
                mensagem_resposta = response.choices[0].message.content
                break # Sai do loop se a chamada for bem-sucedida
            except openai.RateLimitError:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + 1
                    print(f"OpenAI LLM RateLimitError: Tentando novamente em {wait_time} segundos...")
                    time.sleep(wait_time)
                else:
                    mensagem_resposta = "Desculpe, a OpenAI está sobrecarregada no momento. Por favor, tente novamente mais tarde."
                    print("OpenAI LLM RateLimitError: Todas as tentativas de reenvio falharam.")
            except Exception as e:
                print(f"Erro ao chamar OpenAI LLM: {e}")
                mensagem_resposta = "Desculpe, não consegui gerar uma resposta no momento usando OpenAI. Tente novamente."
                break # Sai do loop para outros erros
    else:
        mensagem_resposta = "Modelo de IA não reconhecido. Por favor, escolha 'Gemini' ou 'OpenAI'."

    historico.append({"role": "user", "content": texto})
    historico.append({"role": "assistant", "content": mensagem_resposta})

    return mensagem_resposta, historico

def texto_para_voz(texto: str, idioma: str = "pt") -> str:
    tts = gTTS(text=texto, lang=idioma, slow=False)
    arquivo_temp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tts.save(arquivo_temp.name)
    return arquivo_temp.name

def detectar_idioma_gtts(texto: str) -> str:
    texto_lower = texto.lower()
    if any(p in texto_lower for p in ["the", "is", "are", "hello", "what"]):
        return "en"
    elif any(p in texto_lower for p in ["el", "la", "los", "hola", "que"]):
        return "es"
    elif any(p in texto_lower for p in ["le", "les", "bonjour", "est", "une"]):
        return "fr"
    else:
        return "pt"

# --- Fim das Funções de Serviço ---


st.set_page_config(page_title="Voice AI Assistant", page_icon="🎤", layout="centered")
st.title("🎤 Voice AI Assistant")
st.caption("Fale com seu assistente de IA em qualquer idioma — powered by Whisper + gTTS + Gemini/OpenAI")

# Model selection
st.sidebar.header("Configurações do Modelo")
model_choice = st.sidebar.radio(
    "Escolha o Modelo de IA:",
    ("Gemini", "OpenAI"),
    index=0, # Default to Gemini
    key="llm_model_choice_sidebar"
)

if "historico" not in st.session_state:
    st.session_state.historico = []
if "mensagens_exibidas" not in st.session_state:
    st.session_state.mensagens_exibidas = []

for msg in st.session_state.mensagens_exibidas:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

st.divider()
st.subheader("🎤 Grave sua pergunta")

audio = audiorecorder("▶ Clique para gravar", "■ Clique para parar")

if len(audio) > 0:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
        audio.export(tmp.name, format="wav")
        caminho_audio = tmp.name

    st.audio(caminho_audio, format="audio/wav")

    with st.spinner("🔍 Transcrevendo sua fala..."):
        texto_usuario = transcrever_audio(caminho_audio)

    st.success(f"**Você disse:** {texto_usuario}")

    st.session_state.mensagens_exibidas.append({"role": "user", "content": texto_usuario})

    with st.spinner("🤖 Assistente pensando..."):
        resposta_texto, st.session_state.historico = obter_resposta_llm(
            model_choice,
            texto_usuario,
            st.session_state.historico
        )

    st.session_state.mensagens_exibidas.append({"role": "assistant", "content": resposta_texto})

    with st.chat_message("assistant"):
        st.write(resposta_texto)

    with st.spinner("🔊 Gerando áudio da resposta..."):
        idioma_tts = detectar_idioma_gtts(resposta_texto)
        caminho_resposta_audio = texto_para_voz(resposta_texto, idioma=idioma_tts)

    st.audio(caminho_resposta_audio, format="audio/mp3", autoplay=True)

    os.unlink(caminho_audio)
    os.unlink(caminho_resposta_audio)

st.divider()
if st.button("🗑️ Limpar conversa"):
    st.session_state.historico = []
    st.session_state.mensagens_exibidas = []
    st.rerun()
"""

# Salva o script em um arquivo temporário
with open("streamlit_app.py", "w") as f:
    f.write(streamlit_script)

# --- Garante que qualquer processo streamlit em execução seja encerrado ---
# Pega os PIDs dos processos streamlit em execução
pid_command = !pgrep -f "streamlit run"
pids = [int(p) for p in pid_command if p.isdigit()]

for pid in pids:
    print(f"Encerrando processo Streamlit existente com PID: {pid}")
    !kill {pid}
    # Uma pequena pausa para o processo ser encerrado
    import time
    time.sleep(1)
# ---

# --- Garante que todos os túneis ngrok existentes sejam encerrados ---
print("Encerrando todos os túneis ngrok existentes...")
ngrok.kill()
# ---

# Recupera o token de autenticação do ngrok dos segredos do Colab
ngrok_auth_token = userdata.get('NGROK_API_KEY')

# Define o token de autenticação do ngrok
ngrok.set_auth_token(ngrok_auth_token)

# Inicia o túnel ngrok
public_url = ngrok.connect(8501)
print(f"Seu aplicativo Streamlit estará disponível em: {public_url}")

# Executa o Streamlit em background
!streamlit run streamlit_app.py &>/dev/null&

# Mantém a célula ativa para que o ngrok continue rodando
# Você verá o link do ngrok acima. Clique nele para acessar o app.
# Para parar, interrompa a execução desta célula (botão quadrado de "stop").

2026-04-10 00:14:22.716 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 00:14:22.952 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 00:14:22.954 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 00:14:22.996 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-10 00:14:22.997 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 00:14:22.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 00:14:23.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 00:14:23.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running 

Seu aplicativo Streamlit estará disponível em: NgrokTunnel: "https://tactical-absentee-imagines.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!pgrep -f "streamlit run"

35653


In [ ]:
print("Verificando processos ngrok em execução...")
!ps aux | grep ngrok